## 📌 Introduction

In machine learning, building a model is only the first step. Evaluating and improving the model is crucial to ensure reliable and accurate predictions.

This notebook focuses on evaluating classification models using various performance metrics and improving them using validation and tuning techniques.

Key objectives:
- Evaluate model performance beyond accuracy
- Understand classification metrics
- Improve model reliability using cross-validation
- Optimize model using hyperparameter tuning

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


# 📥 1. Import Libraries

In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)
from sklearn.linear_model import LogisticRegression

# 📂 2. Load Dataset

In [3]:
df = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


# ⚙️ 3. Minimal Preprocessing 

In [4]:
# Handle missing values
df["Age"] = df["Age"].fillna(df["Age"].mean())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

# Feature engineering (reuse logic)
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["Fare_log"] = np.log1p(df["Fare"])

# Encode categorical variables
df["Sex"] = df["Sex"].map({"male": 1, "female": 0})
df["Embarked"] = df["Embarked"].map({"S": 0, "C": 1, "Q": 2})

# Features
features = ["Pclass", "Age", "Fare_log", "FamilySize", "Sex", "Embarked"]
X = df[features]
y = df["Survived"]

print("Features ready:", X.shape)

Features ready: (891, 6)


# 🔀 4. Train-Test Split

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (712, 6)
Test shape: (179, 6)


# 🤖 5. Train Model

In [6]:
model = LogisticRegression(max_iter=200)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Model trained successfully")

Model trained successfully


# 📊 6. Evaluation Metrics (CORE SECTION 🔥)


In [7]:
print("📊 Model Evaluation Metrics")
print("-" * 30)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))

📊 Model Evaluation Metrics
------------------------------
Accuracy : 0.8044692737430168
Precision: 0.782608695652174
Recall   : 0.7297297297297297
F1 Score : 0.7552447552447552


# 📉 7. Confusion Matrix

In [8]:
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:\n", cm)

Confusion Matrix:
 [[90 15]
 [20 54]]


### 📊 Confusion Matrix Interpretation

- True Positives (TP): Correctly predicted survival  
- True Negatives (TN): Correctly predicted non-survival  
- False Positives (FP): Incorrectly predicted survival  
- False Negatives (FN): Missed survival cases  

This helps understand where the model makes mistakes.

# 🔁 8. Cross Validation (VERY IMPORTANT)

In [9]:
cv_scores = cross_val_score(model, X, y, cv=5)

print("Cross Validation Scores:", cv_scores)
print("Mean CV Score:", cv_scores.mean())

Cross Validation Scores: [0.77653631 0.78089888 0.7752809  0.78651685 0.81460674]
Mean CV Score: 0.7867679367271359


# ⚙️ 9. Hyperparameter Tuning

In [10]:
params = {"C": [0.1, 1, 10]}

grid = GridSearchCV(LogisticRegression(max_iter=200), param_grid=params, cv=5)
grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
print("Best CV Score:", grid.best_score_)

Best Parameters: {'C': 10}
Best CV Score: 0.7934699103713188


# 📈 11. Compare Before vs After

In [11]:
# Before tuning
print("Before Tuning Accuracy:", accuracy_score(y_test, y_pred))

# After tuning
best_model = grid.best_estimator_
y_pred_new = best_model.predict(X_test)

print("After Tuning Accuracy:", accuracy_score(y_test, y_pred_new))

Before Tuning Accuracy: 0.8044692737430168
After Tuning Accuracy: 0.8044692737430168


## 🧠 Key Insights

- Accuracy alone is not sufficient to evaluate a model  
- Precision and Recall provide deeper understanding of performance  
- Cross-validation improves reliability of results  
- Hyperparameter tuning enhances model performance  

## 🏁 Conclusion

Model evaluation and optimization are critical steps in machine learning.  
Using proper evaluation metrics and tuning techniques ensures that models are reliable and perform well on unseen data.

This process transforms a basic model into a production-ready solution.